# Let's build GPT: from scratch, in code, spelled out.

In [22]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [23]:
len(text)

1115394

In [24]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [25]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [26]:
stoi = {s : i for i,s in enumerate(chars)}
itos = {i : s for i,s in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda i: ''.join([itos[c] for c in i])

enkod = encode("belal")
print(enkod)
print(decode(enkod))

[40, 43, 50, 39, 50]
belal


In [27]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
data.shape, data.dtype

(torch.Size([1115394]), torch.int64)

In [28]:
data[:100]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])

In [29]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [30]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data)-block_size ,(batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb,yb = get_batch("train")
xb,yb

(tensor([[24, 43, 58,  5, 57,  1, 46, 43],
         [44, 53, 56,  1, 58, 46, 39, 58],
         [52, 58,  1, 58, 46, 39, 58,  1],
         [25, 17, 27, 10,  0, 21,  1, 54]]),
 tensor([[43, 58,  5, 57,  1, 46, 43, 39],
         [53, 56,  1, 58, 46, 39, 58,  1],
         [58,  1, 58, 46, 39, 58,  1, 46],
         [17, 27, 10,  0, 21,  1, 54, 39]]))

#### simplest baseline: bigram language model, loss, generation

In [31]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, target=None):
        logits = self.token_embedding_table(idx)
        if(target == None):
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            target = target.view(B*T)
            loss = F.cross_entropy(logits, target)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:,-1,:]
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx,next_token), dim=1)
        return idx

model = BigramLanguageModel(vocab_size=vocab_size)
idx = torch.randint(0,vocab_size,(batch_size,block_size))
target = torch.randint(0,vocab_size,(batch_size,block_size))
logits,loss = model(xb, yb)
print(logits.shape, loss)

idx2 = torch.zeros((1,1), dtype=torch.long)
print(decode(model.generate(idx2, 100)[0].tolist()))

torch.Size([32, 65]) tensor(4.8786, grad_fn=<NllLossBackward0>)

,pw3Q
p$lst;IN.P,VBgKHnxEdd&yFWKEr;R'KIqw-QA!ajER
tN
g.olOdi.Oszx;gRSqVk f?YQHZ;vv$v$J;bmNL&lvN!a!qH


#### training the bigram model

In [32]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [33]:
batch_size = 32

for _ in range(10000):
    xb, yb = get_batch("train")
    logits, loss = model(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.5165133476257324


In [34]:
print(decode(model.generate(torch.zeros((1,1), dtype=torch.long), 100)[0].tolist()))


Firden.

Yond fe hs,
GAn hthe, e d tar ithe?

Whe l y mer, on s wist wis ncre hath aryr titr;
meal T


#### version 1: averaging past context with for loops, the weakest form of aggregation

In [35]:
torch.manual_seed(1337)
B,T,C = 4,8,2
x = torch.randn(B,T,C)

In [36]:
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b,t] = torch.mean(xprev, 0)

#### version 2: using matrix multiply

In [39]:
w = torch.tril(torch.ones(T,T))
w = w / w.sum(1, keepdim=True)
xbow2 = w @ x
torch.allclose(xbow,xbow2)

False

In [43]:
print(xbow[3])
print(xbow2[3])

tensor([[ 1.6455, -0.8030],
        [ 1.4985, -0.5395],
        [ 0.4954,  0.3420],
        [ 1.0623, -0.1802],
        [ 1.1401, -0.4462],
        [ 1.0870, -0.4071],
        [ 1.0430, -0.1299],
        [ 1.1138, -0.1641]])
tensor([[ 1.6455, -0.8030],
        [ 1.4985, -0.5395],
        [ 0.4954,  0.3420],
        [ 1.0623, -0.1802],
        [ 1.1401, -0.4462],
        [ 1.0870, -0.4071],
        [ 1.0430, -0.1299],
        [ 1.1138, -0.1641]])


#### version 3: adding softmax

In [46]:
tril = torch.tril(torch.ones(T,T))
w = torch.zeros((T,T))
w = w.masked_fill(tril==0,float('-inf'))
w = F.softmax(w, dim=-1)
xbow3 = w @ x
torch.allclose(xbow2,xbow3)

True